# 17 - AdTech: Consent as Three Permissions in One Field

A single `marketing_consent = true` collapses three distinct
permissions. A person may permit measurement but not
personalization, or activation but not cross-context sharing —
an agent reading one boolean treats all uses as allowed, which
is a compliance violation, not a data error.

The estate keeps the legacy boolean AND the fixed model: three
purpose-scoped consent bases, served per declared purpose.

| Declared purpose | Consent basis | Lane |
| --- | --- | --- |
| `analytics.reporting` (measurement) | `consent_measurement` | consent question |
| `marketing.personalization` | `consent_personalization` | consent question |
| `sharing.third_party` (activation) | `consent_activation` | consent + transfer |

Metatate never reads data values: each answer is a typed
`consent_required` condition naming the exact basis column the
agent must verify — never satisfiable by caller assertion.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the Customer 360 demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


PRODUCT_DATABASE_TABLES = {"product_usage_events", "support_tickets", "ml_feature_store"}


def asset(table, column=None, schema="public", database=None):
    resolved_database = database or (
        "product" if table in PRODUCT_DATABASE_TABLES else "master"
    )
    ref = {"database": resolved_database, "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## 1. One field, three questions


In [ ]:
consent_cases = [
    ("measurement", dict(
        asset=asset("customers"),
        use="measure campaign reach across customers",
        scenario_key="consent.required",
        purpose_key="analytics.reporting",
    )),
    ("personalization", dict(
        asset=asset("customers"),
        use="personalize offers from customer records",
        scenario_key="consent.required",
        purpose_key="marketing.personalization",
    )),
    ("activation", dict(
        asset=asset("customers"),
        use="activate an audience with a demand-side partner",
        scenario_key="consent.required",
        purpose_key="sharing.third_party",
    )),
]

consents = {}
for label, arguments in consent_cases:
    answer = client.authorize_use(**arguments)
    consents[label] = answer
    condition = next(
        (c for c in answer.get("conditions", [])
         if c.get("kind") == "consent_required"),
        {},
    )
    projection = condition.get("projection") or {}
    print(
        f"{label:16} -> {answer_label(answer):12} "
        f"verify {projection.get('basis_column', '?')}"
    )


Same record, three different verification requirements — the
purpose selects the consent basis, and the typed projection
names the exact column the agent must check before proceeding.


## 2. Consent recorded is not use permitted


In [ ]:
allowed_lane = client.authorize_use(
    asset("customers"),
    use="personalize offers from customer records",
    scenario_key="purpose.allowed_use",
    purpose_key="marketing.personalization",
)
prohibited_lane = client.authorize_use(
    asset("customers"),
    use="personalize offers from customer records",
    scenario_key="purpose.prohibited_use",
    purpose_key="marketing.personalization",
)
print("allowed-use lane    ->", answer_label(allowed_lane))
print("prohibited-use lane ->", answer_label(prohibited_lane))


Personalization of the raw record is not merely
consent-conditioned — no permit covers it (review) and the
prohibition names it outright (deny). Consent and permission
are separate questions with separate answers.


## 3. Missing purpose fails closed


In [ ]:
missing_purpose = client.authorize_use(
    asset("customers"),
    use="use customer records for an advertising workflow",
    scenario_key="consent.required",
)
print("missing purpose ->", missing_purpose["state"], missing_purpose["reason_code"])


## 4. Activation is a transfer


In [ ]:
approved = client.authorize_use(
    asset("ad_audience_exports"),
    use="activate the consented audience with an approved partner",
    scenario_key="residency.cross_border_transfer",
    operation="export",
    destination={"system": "APPROVED_DSP_A"},
)
unlisted = client.authorize_use(
    asset("ad_audience_exports"),
    use="activate the audience with an unlisted platform",
    scenario_key="residency.cross_border_transfer",
    operation="export",
    destination={"system": "UNLISTED_DSP"},
)
print("approved partner ->", answer_label(approved))
print("unlisted partner ->", answer_label(unlisted))


## 5. The purpose flips the verdict, not the SQL


In [ ]:
CONSENT_SQL = "SELECT customer_id, consent_measurement, consent_personalization, consent_activation FROM customers"

measurement_sql = client.validate_query_context(
    CONSENT_SQL,
    scenario_key="purpose.allowed_use",
    default_database="master", default_schema="public",
    purpose_key="analytics.reporting",
)
personalization_sql = client.validate_query_context(
    CONSENT_SQL,
    scenario_key="purpose.allowed_use",
    default_database="master", default_schema="public",
    purpose_key="marketing.personalization",
)
print("measurement     ->", measurement_sql.get("verdict"), "/", measurement_sql.get("state"))
print("personalization ->", personalization_sql.get("verdict"), "/", personalization_sql.get("state"))


## 6. The receipt


In [ ]:
receipt = client.explain_why(
    authorization_id=consents["measurement"]["authorization_id"],
)
print("decision  :", receipt["decision"], "/", receipt["answer_state"])
print("cited rows:", len(receipt["cited_decision_ids"]))
print("evaluated :", receipt["provenance"]["evaluated_at"])


Byte-identical SQL, two verdicts — the declared purpose is the
decision-bearing input. And every determination above is a
durable, citable record: the receipt reconstructs what was
asked, which policy answered, and why.
